# Adding a new model to the benchmark

This notebook guides you through the process of adding a new model to the benchmark.
Specifically, it shows how PRECISE was added to the benchmark during the review process of this paper.

PRECISE is an external package, so there is no model code to write and we only show the benchmark code that every new method needs.

**How to use this notebook**

- Every step has an explanation and the change as a `diff`. The green `+` lines are what you would type. Open the named file next to the notebook to see the surrounding context.
- Nothing here trains a model on real data or modifies the repository. It shows you the steps, not an output.
- Two parts contain code you can run. In the sandbox (Step 2f) you can plug in your own model and check that it takes the inputs and returns the outputs the benchmark expects. The checklist in the last section scans the repository for a model name and lists the edits that are still missing.

To add a model you write one function and register its name in a few places:

| Step | Where | What |
|---|---|---|
| 0 | environment or `code/frameworks/` | make the method importable |
| 1, 2 | `code/training_utils.py` | `run_<model>_benchmark` with the expected inputs and outputs |
| 3 | `code/hyper_tuning.py` | import, `<Model>Args`, Optuna search-space branch, default parameters |

## Step 0: Make the method importable

There are two ways to do this, depending on how the method is distributed.

**Route A: the method is an installable package.** This is the case for PRECISE. Nothing goes into `code/frameworks/`. You install the package into the benchmark environment and document the command in the README:

```bash
pip install --no-deps "git+https://github.com/NKI-CCB/PRECISE.git@f1b1399b85a229e65b2fb33a48d97c8b0ee4eec6"
```

`environment.yaml` does not install PRECISE. It is a manual step described in the README. If you add a pip-installed method, add its install command to the README in the same place.

**Route B: the method is research code that you copy into the repository.** This is the case for SCAD, scDEAL, SSDA4Drug and scATD. Create `code/frameworks/<YourMethod>/` following the existing layout:

```
SCAD/        callbacks.py  lightning_datamodule.py  lightning_module.py  main.py  modules.py
SSDA4Drug/                 lightning_datamodule.py  lightning_module.py  main.py  modules.py
scATD/                     lightning_datamodule.py  lightning_module.py  main.py  modules.py  pretrained_models/
scDeal/                    lightning_datamodule.py  lightning_module.py  main.py  modules.py  scDEAL_utils.py  DaNN/
```

The convention is `modules.py` (building blocks), `lightning_module.py` (training logic), `lightning_datamodule.py` (data loaders) and `main.py` (stand-alone entry point). The benchmark never calls `main.py`. It imports the module classes from inside the runner function.

## Step 1: Understand what your runner receives and what it must return

### 1a. What it receives

You do **not** load or preprocess data yourself. `prepare_data` does it identically for every method, which keeps the comparison fair. Your runner receives the result as keyword arguments. This is the dict (end of `prepare_data` in `hyper_tuning.py`):

```python
    return {
        "x_train_source": X_source_train,
        "y_train_source": y_source_train,
        "x_val_source": X_source_val,
        "y_val_source": y_source_val,
        "x_test_source": X_source_test,
        "y_test_source": y_source_test,
        "x_train_target": X_target_train,
        "y_train_target": y_target_train,
        "x_test_target": X_target_test,
        "y_test_target": y_target_test,
        "X_target_independent": None,        # only filled for cross-dataset evaluation
        "y_target_independent": None,
        "target_file": target_tag,
        "independent_target_file": "independent_placeholder",
    }
```

| key | what it is |
|---|---|
| `x_train_source`, `x_val_source`, `x_test_source` | bulk cell-line expression, `DataFrame` of samples x genes (64 / 16 / 20 % split) |
| `y_train_source`, `y_val_source`, `y_test_source` | source response, continuous in [0, 1] (`1 - viability`) |
| `x_train_target`, `x_test_target` | single-cell expression of the target dataset (80 / 20 % split) |
| `y_train_target`, `y_test_target` | target labels, binary |
| `X_target_independent`, `y_target_independent` | a second, unseen target cohort; `None` during hyperparameter tuning |
| `target_file`, `independent_target_file` | dataset tags (strings), for logging |

Before you write any code, note that:

1. The data is already standardized. A `StandardScaler` is fit on the source training split and applied to every split, and all matrices share one gene order. If your method centers or scales its input internally, switch that off. PRECISE, for example, is run with `mean_center=False, std_unit=False`.
2. Source labels are continuous. All methods binarize them at `>= 0.5` to compute metrics. Whether you train on the continuous or the binarized value is up to you. The neural models expose this as the tunable `binarize_source`, and PRECISE regresses on the continuous value.
3. You may use `x_train_target` for unsupervised alignment, since this is what domain adaptation methods do. `y_train_target` is reserved for the few-shot models. Nothing with `test` or `independent` in its name may influence training or threshold selection.

### 1b. What it must return

Every runner returns a dict with the same five keys:

```python
    return {
        "source_val": source_val_metrics,
        "source_test": source_test_metrics,
        "target_test": target_test_metrics,
        "independent_target_test": independent_metrics,
        "selected_threshold": tuned_threshold,
    }
```

- `source_val`, `source_test`, `target_test` and `independent_target_test` are metric dicts produced by the shared helper `calculate_all_metrics(y_true, y_pred_proba, threshold)`. It returns `auc`, `auprc`, `accuracy`, `precision`, `recall_sensitivity`, `f1_score`, `specificity` and `mcc`. Use this helper and do not compute metrics yourself. It fixes the metric names and handles edge cases the same way for all methods (single-class cohorts give `-1`, NaN predictions are replaced).
- `independent_target_test` is an empty dict `{}` when no independent cohort was passed, i.e. during tuning.
- `selected_threshold` is the decision threshold you tuned, as a float.

The key names are used later on. `hyper_tuning.py` flattens the dict into W&B keys of the form `f"{split}_{metric}"`, e.g. `source_val_mcc`. Optuna maximizes `source_val_mcc`, and the best trial is later selected by reading that key back from W&B.

## Step 2: Write the runner function in `training_utils.py`

This is the only substantial code you write. The diffs below show `run_precise_benchmark` in five parts. For your own method, 2a and 2e stay almost the same, while 2b to 2d depend on the method.

### 2a. Signature

Copy the signature as it is and change only the function name.

```diff
+def run_precise_benchmark(
+    args,                       # dict of hyperparameters, vars(PreciseArgs), see Step 3
+    x_train_source, y_train_source,
+    x_val_source, y_val_source,
+    x_test_source, y_test_source,
+    x_train_target, y_train_target,
+    x_test_target, y_test_target,
+    X_target_independent, y_target_independent,
+    target_file, independent_target_file,
+    seed=42,
+    trial=None,                 # the Optuna trial, only needed for pruning
+):
+    """Run PRECISE as a classical source-to-target domain adaptation baseline."""
```

`hyper_tuning.py` calls the runner as `benchmark_func(args_dict, **data, trial=trial, seed=split_seed)`. `args` therefore has to be the first positional parameter, and every data key has to appear under exactly that name, including the ones your method does not use.

### 2b. Set the seed and check the inputs

What you check here depends on your method. PRECISE needs at least two source samples per cross-validation fold and cannot use more factors than there are samples, so the code limits both to what the data allows:

```diff
+    set_seed(seed)
+
+    n_factors = int(args.get("n_factors", 70))
+    n_pv = int(args.get("n_pv", 40))
+    requested_cv_fold = int(args.get("cv_fold", 10))
+    effective_cv_fold = min(requested_cv_fold, x_train_source.shape[0])
+    if effective_cv_fold < 2:
+        raise ValueError("PRECISE needs at least two source samples for cross-validation.")
+    min_source_cv_train = x_train_source.shape[0] - int(np.ceil(x_train_source.shape[0] / effective_cv_fold))
+    max_factors = min(min_source_cv_train, x_train_target.shape[0], x_train_source.shape[1])
+    if max_factors < 2:
+        raise ValueError("PRECISE needs at least two available factors after source/target alignment.")
+    effective_n_factors = min(n_factors, max_factors)
+    effective_n_pv = min(n_pv, effective_n_factors)
```

Three things in this block apply to any method:

- Read hyperparameters with `args.get(key, default)`. When a configuration is read back from W&B it contains extra keys such as `split`, and `.get` ignores them.
- Limit hyperparameters to what the data allows. The benchmark contains 28 datasets and some are small, so a value that works on one dataset (70 PCA factors) can be impossible on another (40 cells). Log the value that was used to `wandb.config`. `run_precise_benchmark` does this right after the block above.
- Raise a `ValueError` for inputs your method cannot handle. `hyper_tuning.py` catches `ValueError`, logs it and scores the trial `-1`. Any other exception type aborts the whole sweep.

### 2c. Build and fit the model

Fit on the source training data and its labels. If your method adapts to the target domain it may also use `x_train_target`, but no target labels.

```diff
+    use_data = bool(args.get("use_data", True))
+    model = DrugResponsePredictor(
+        n_representations=int(args.get("n_representations", 100)),
+        method=args.get("method", "consensus"),
+        mean_center=bool(args.get("mean_center", False)),     # data is already standardised upstream
+        std_unit=bool(args.get("std_unit", False)),
+        n_factors=effective_n_factors,
+        n_pv=effective_n_pv,
+        dim_reduction=args.get("dim_reduction", "pca"),
+        dim_reduction_target=args.get("dim_reduction_target", None),
+        l1_ratio=float(args.get("l1_ratio", 0.0)),
+        source_data=None if use_data else x_train_source.to_numpy(dtype=np.float64),
+        target_data=x_train_target.to_numpy(dtype=np.float64),   # target expression, no labels
+        n_jobs=int(args.get("n_jobs", 1)),
+    )
+    model.cv_fold = effective_cv_fold
+    model.verbose = int(args.get("verbose", 0))
+    model.fit(
+        x_train_source.to_numpy(dtype=np.float64),
+        _y_to_vector(y_train_source).astype(np.float64),          # continuous source response
+        use_data=use_data,
+    )
```

`_y_to_vector` is a shared helper that flattens a `DataFrame`, `Series` or array of labels to 1-D.

### 2d. Provide `predict_proba`

The shared threshold tuner expects an sklearn-style `model.predict_proba(X)[:, 1]`. If your model already has this method, skip this part. If it returns logits, scores or rankings, wrap it in a small adapter. PRECISE is a regressor that returns an unbounded score, so its adapter clips the score to [0, 1]:

```diff
+    class _PreciseProbAdapter:
+        def __init__(self, fitted_model):
+            self.fitted_model = fitted_model
+
+        def predict_proba(self, X):
+            values = X.to_numpy(dtype=np.float64) if hasattr(X, "to_numpy") else np.asarray(X, dtype=np.float64)
+            scores = np.asarray(self.fitted_model.predict(values)).ravel()
+            scores = np.nan_to_num(scores, nan=0.5, posinf=1.0, neginf=0.0)
+            scores = np.clip(scores, 0.0, 1.0)
+            return np.column_stack([1.0 - scores, scores])
+
+    adapter = _PreciseProbAdapter(model)
```

### 2e. Tune the threshold on source validation and compute the metrics

```diff
+    tuned_threshold, _ = _tune_threshold_GB(adapter, x_val_source, y_val_source)
+
+    source_val_metrics = calculate_all_metrics(
+        _y_to_vector(y_val_source >= 0.5),
+        adapter.predict_proba(x_val_source)[:, 1],
+        threshold=tuned_threshold,
+    )
+    # same call for (x_test_source, y_test_source) and (x_test_target, y_test_target)
+
+    independent_metrics: Dict[str, float] = {}
+    if X_target_independent is not None and y_target_independent is not None:
+        independent_metrics = calculate_all_metrics(
+            _y_to_vector(y_target_independent >= 0.5),
+            adapter.predict_proba(X_target_independent)[:, 1],
+            threshold=tuned_threshold,
+        )
+
+    return {
+        "source_val": source_val_metrics,
+        "source_test": source_test_metrics,
+        "target_test": target_test_metrics,
+        "independent_target_test": independent_metrics,
+        "selected_threshold": tuned_threshold,
+    }
```

The threshold is chosen on `source_val` only, which is also the split that Optuna optimizes. No target label may influence model selection, otherwise the comparison between methods is not fair. There are two tuners: `_tune_threshold_GB(model, X_val, y_val)` for anything with `predict_proba`, and `_tune_threshold_DL_source_only(get_preds, val_loader)` for Lightning models that use dataloaders.

The finished function is `run_precise_benchmark` in `code/training_utils.py`.

### 2f. Sandbox: test your runner before changing the repository

The next cells build a small synthetic dataset with the same keys and types as the real one (continuous source labels, binary target labels, standardized features) and define a skeleton runner. `check_runner` then calls the runner twice, once with an independent cohort and once with `None` as in `hyper_tuning.py`, and checks the returned dict.

To test your own method, replace the three lines marked `# <<<` and re-run the cells. When the check passes, paste the function into `training_utils.py`.

The first cell imports `training_utils`, which loads torch, lightning, catboost and precise and takes a moment. If the import fails, the message names the missing dependency.

In [ ]:
import numpy as np, pandas as pd
from training_utils import calculate_all_metrics, _tune_threshold_GB, _y_to_vector, set_seed


def make_toy_bundle(n_source=150, n_target=80, n_genes=30, seed=0, with_independent=True):
    "Synthetic stand-in for prepare_data(): same keys, same types, tiny."
    rng = np.random.default_rng(seed)
    genes = [f"ENSG{i:011d}" for i in range(n_genes)]
    w = rng.normal(size=n_genes)

    def cohort(n, prefix, shift):
        X = pd.DataFrame(rng.normal(loc=shift, size=(n, n_genes)), columns=genes,
                         index=[f"{prefix}{i}" for i in range(n)])
        return X, pd.Series(1 / (1 + np.exp(-(X.to_numpy() @ w) / 2)), index=X.index, name="response")

    Xs, ys = cohort(n_source, "cellline_", 0.0)           # source: continuous response in [0, 1]
    Xt, yt = cohort(n_target, "cell_", 0.4)               # target: domain-shifted, binary labels
    Xi, yi = cohort(n_target, "indep_", -0.4)
    yt, yi = (yt >= 0.5).astype(int), (yi >= 0.5).astype(int)
    a, b, c = int(0.64 * n_source), int(0.8 * n_source), int(0.8 * n_target)
    return {
        "x_train_source": Xs[:a], "y_train_source": ys[:a],
        "x_val_source": Xs[a:b], "y_val_source": ys[a:b],
        "x_test_source": Xs[b:], "y_test_source": ys[b:],
        "x_train_target": Xt[:c], "y_train_target": yt[:c],
        "x_test_target": Xt[c:], "y_test_target": yt[c:],
        "X_target_independent": Xi if with_independent else None,
        "y_target_independent": yi if with_independent else None,
        "target_file": "TOY_TARGET", "independent_target_file": "TOY_INDEPENDENT",
    }


bundle = make_toy_bundle()
for k, v in bundle.items():
    print(f"{k:26s}", type(v).__name__, getattr(v, "shape", v))

x_train_source             DataFrame (96, 30)
y_train_source             Series (96,)
x_val_source               DataFrame (24, 30)
y_val_source               Series (24,)
x_test_source              DataFrame (30, 30)
y_test_source              Series (30,)
x_train_target             DataFrame (64, 30)
y_train_target             Series (64,)
x_test_target              DataFrame (16, 30)
y_test_target              Series (16,)
X_target_independent       DataFrame (80, 30)
y_target_independent       Series (80,)
target_file                str TOY_TARGET
independent_target_file    str TOY_INDEPENDENT


In [2]:
from sklearn.linear_model import LogisticRegression


def run_mymodel_benchmark(
    args,
    x_train_source, y_train_source, x_val_source, y_val_source, x_test_source, y_test_source,
    x_train_target, y_train_target, x_test_target, y_test_target,
    X_target_independent, y_target_independent,
    target_file, independent_target_file,
    seed=42, trial=None,
):
    "Skeleton runner. Swap the model; keep everything else."
    set_seed(seed)

    model = LogisticRegression(C=float(args.get("C", 1.0)), max_iter=1000, random_state=seed)   # <<< your model
    y_fit = (_y_to_vector(y_train_source) >= 0.5).astype(int)                                   # <<< or keep it continuous
    model.fit(x_train_source, y_fit)                                                            # <<< may also use x_train_target (no labels)

    # Needs model.predict_proba; otherwise wrap the model in an adapter as in Step 2d.
    tuned_threshold, _ = _tune_threshold_GB(model, x_val_source, y_val_source)                  # source-val only

    def score(X, y):
        return calculate_all_metrics(_y_to_vector(y >= 0.5), model.predict_proba(X)[:, 1], threshold=tuned_threshold)

    independent = {}
    if X_target_independent is not None and y_target_independent is not None:
        independent = score(X_target_independent, y_target_independent)

    return {
        "source_val": score(x_val_source, y_val_source),
        "source_test": score(x_test_source, y_test_source),
        "target_test": score(x_test_target, y_test_target),
        "independent_target_test": independent,
        "selected_threshold": tuned_threshold,
    }

In [ ]:
METRICS = {"auc", "auprc", "accuracy", "precision", "recall_sensitivity", "f1_score", "specificity", "mcc"}


def check_runner(runner, args):
    "Call the runner the way the benchmark does, and check what comes back."
    for label, with_independent in [("with an independent cohort", True), ("hyper_tuning.py style (independent=None)", False)]:
        res = runner(dict(args), **make_toy_bundle(with_independent=with_independent), trial=None, seed=42)
        assert set(res) == {"source_val", "source_test", "target_test", "independent_target_test", "selected_threshold"}, set(res)
        for split in ("source_val", "source_test", "target_test"):
            assert set(res[split]) == METRICS, (split, set(res[split]) ^ METRICS)
        assert (set(res["independent_target_test"]) == METRICS) if with_independent else (res["independent_target_test"] == {})
        assert 0.0 < float(res["selected_threshold"]) < 1.0
        print(f"PASS  {label}: source_val mcc={res['source_val']['mcc']:.2f}  target_test mcc={res['target_test']['mcc']:.2f}")

check_runner(run_mymodel_benchmark, {"C": 1.0})

PASS  with an independent cohort: source_val mcc=0.92  target_test mcc=0.77


PASS  hyper_tuning.py style (independent=None): source_val mcc=0.92  target_test mcc=0.77


## Step 3: Register the model in `hyper_tuning.py`

There are four edits. `--model` has no `choices=` list to extend. An unknown name ends in `raise ValueError(f"Unknown model: {model_name}")`.

### 3a. Import the runner

```diff
 from training_utils import (  # noqa: E402
     run_scatd_benchmark,
+    run_precise_benchmark,
     run_catboost_benchmark,
```

### 3b. Add an `Args` class with every setting of the method and its default

```diff
+class PreciseArgs:
+    def __init__(self, drug_name):
+        self.drug = drug_name
+        self.data_split = "test"
+        self.mode = "PRECISE"
+        self.n_factors = 70
+        self.n_pv = 40
+        self.n_representations = 100
+        self.method = "consensus"
+        self.mean_center = False
+        self.std_unit = False
+        self.dim_reduction = "pca"
+        self.dim_reduction_target = None
+        self.l1_ratio = 0.0
+        self.n_jobs = 1
+        self.cv_fold = 10
+        self.verbose = 0
+        self.use_data = True
```

It is a plain class because the script converts it to the runner's `args` dict with `vars(args)`. Include the settings you do not tune as well (here `n_factors` and `n_pv`), so that the class records the full configuration that was run.

### 3c. Add the search-space branch inside `objective(trial)`

```diff
                     elif model_name == "scATD":
                         ...
                         benchmark_func = run_scatd_benchmark

+                    elif model_name == "PRECISE":
+                        args = PreciseArgs(drug)
+                        args.method = trial.suggest_categorical("method", ["consensus", "elasticnet"])
+                        args.n_representations = trial.suggest_categorical("n_representations", [50, 100, 150])
+                        if args.method == "elasticnet":
+                            args.l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 0.5, 1.0])
+                        else:
+                            args.l1_ratio = 0.0
+                        benchmark_func = run_precise_benchmark
```

A branch creates the defaults, overwrites the tuned attributes with `trial.suggest_*` and sets `benchmark_func`.

The name passed to `suggest_*` has to equal the attribute name, e.g. `"method"` and `args.method`. `trial.params` becomes the W&B run config, and tuned values are later applied to the args by that name. If the names differ, the tuned value is never applied and no error is raised.

The search space can be conditional. For PRECISE, `l1_ratio` is only tuned when `method` is `elasticnet`. Keep the search space similar in size to those of the other methods, because every model gets the same `--n_trials` budget.

### 3d. Define the method's default parameters for tuning

```diff
                 elif model_name == "scATD":
                     initial_params = {...}
+                elif model_name == "PRECISE":
+                    initial_params = {
+                        "method": "consensus",
+                        "n_representations": 100,
+                    }
```

`study.enqueue_trial(initial_params)` makes trial 0 the default configuration of the method's authors. The published setup is then always part of the results, and tuning can only improve on it.

### 3e. Code you do not need to edit

The code after the branch is the same for all models. It defines how your runner is called and what gets logged (condensed from `objective` in `hyper_tuning.py`):

```python
args_dict = vars(args)
wandb_config = dict(trial.params)          # the suggest_* names become the W&B run config
wandb_config["split"] = split

with wandb.init(project=WANDB_PROJECT, group=f"{drug}_{target_tag}_{model_name}_split{split}",
                job_type="optuna_trial", config=wandb_config, reinit=True) as run:
    try:
        if model_name in RANDOM_CELL_SEED_MODELS:
            ...                             # few-shot models: average over 3 random-cell seeds
        elif model_name.startswith("CatBoost"):
            results = benchmark_func(**args_dict, **data, trial=trial, seed=split_seed)
        else:
            results = benchmark_func(args_dict, **data, trial=trial, seed=split_seed)
    except ValueError as exc:
        run.log({"error": f"{model_name} failed: {exc}"})
        return -1.0

    run.log({"drug": drug, "target": target_tag, "model_name": model_name, "split": split,
             **{f"{split}_{metric}": value                      # e.g. source_val_mcc, target_test_auc
                for split, metrics in results.items() if isinstance(metrics, dict)
                for metric, value in metrics.items()}})
    return results.get("source_val", {}).get("mcc", -1.0)      # what Optuna maximises
```

- Models other than CatBoost are called as `benchmark_func(args_dict, **data, trial=trial, seed=split_seed)`, which is why the signature in Step 2a is fixed.
- `RANDOM_CELL_SEED_MODELS` contains the few-shot models. They draw random target cells, so each trial is averaged over three seeds. Add your model to this set only if it samples labelled target cells. PRECISE does not.
- There are three splits with `split_seed = SEED + split - 1`, i.e. 42, 43 and 44, and one Optuna study per split. Your runner receives this value as `seed`.
- Metrics are logged to W&B as `source_val_mcc`, `target_test_auc` and so on, in the run group `f"{drug}_{target_tag}_{model_name}_split{split}"`.
- Optuna maximizes `results["source_val"]["mcc"]`. A `ValueError` from your runner scores `-1.0`.

## Step 4: Smoke-test on the demo dataset

Before submitting a sweep over all 17 drugs, run two trials on the demo drug. `WANDB_MODE=offline` keeps these test runs out of your W&B project:

```bash
conda activate benchmark
WANDB_MODE=offline python code/hyper_tuning.py --drugs SN-38 --n_trials 2 --model <YourModel>
```

Three studies (splits 1 to 3) should start and finish, each trial should print a `source_val` MCC and not `<YourModel> failed: ...`, and trial 0 should use your `initial_params`. If every trial scores `-1.0`, your runner raises a `ValueError`. The printed message says why.

## Integration checklist

This cell scans the repository and reports which of the edits from this notebook are in place for a given model. It is set to PRECISE. Change the three names to your own model and re-run it as you work through the steps. The integration is complete when every required row says `ok`.

In [4]:
import re
from pathlib import Path

REPO = Path.cwd().parent            # this notebook lives in code/

MODEL  = "PRECISE"                  # the string passed to --model / --models
RUNNER = "run_precise_benchmark"    # function in code/training_utils.py
ARGS   = "PreciseArgs"              # args container class
FRAMEWORK_DIR = None                # e.g. "MyMethod" if your code lives in code/frameworks/MyMethod/, else None


def text(rel):
    p = REPO / rel
    return p.read_text() if p.exists() else ""

tu, ht = text("code/training_utils.py"), text("code/hyper_tuning.py")
branch = lambda src: len(re.findall(rf'model_name == "{re.escape(MODEL)}"', src))

checks = [  # (step, required, description, passed)
    ("0",  bool(FRAMEWORK_DIR), f"code/frameworks/{FRAMEWORK_DIR}/ exists", bool(FRAMEWORK_DIR) and (REPO / "code/frameworks" / str(FRAMEWORK_DIR)).is_dir()),
    ("2",  True,  f"training_utils.py defines {RUNNER}()", f"def {RUNNER}(" in tu),
    ("2",  True,  "  and returns the 5-key result dict", bool(re.search(rf"def {RUNNER}\(.*?\"selected_threshold\"", tu, re.S))),
    ("3a", True,  f"hyper_tuning.py imports {RUNNER}", RUNNER in ht),
    ("3b", True,  f"hyper_tuning.py defines class {ARGS}", f"class {ARGS}" in ht),
    ("3c", True,  "hyper_tuning.py has an Optuna search-space branch", branch(ht) >= 1),
    ("3d", False, "hyper_tuning.py enqueues initial_params (warm start)", branch(ht) >= 2),
]

print(f"Integration status for {MODEL!r}\n")
for step, required, desc, ok in checks:
    if not required and not ok and step == "0":
        continue                                  # pip-installed method: no framework folder expected
    mark = "ok" if ok else ("MISSING" if required else "optional")
    print(f"  {mark:9s} Step {step:3s} {desc}")
missing = [d for _, req, d, ok in checks if req and not ok]
print(f"\n{len(missing)} required edit(s) missing." if missing else "\nAll required edits are in place.")

Integration status for 'PRECISE'

  ok        Step 2   training_utils.py defines run_precise_benchmark()
  ok        Step 2     and returns the 5-key result dict
  ok        Step 3a  hyper_tuning.py imports run_precise_benchmark
  ok        Step 3b  hyper_tuning.py defines class PreciseArgs
  ok        Step 3c  hyper_tuning.py has an Optuna search-space branch
  ok        Step 3d  hyper_tuning.py enqueues initial_params (warm start)

All required edits are in place.


## Recap

1. Make the method importable, either with a pinned `pip install --no-deps` or as a folder under `code/frameworks/`.
2. Write one function in `training_utils.py` with the fixed signature and the five return keys. Read hyperparameters via `args.get`, raise `ValueError` for inputs the method cannot handle, use target data without labels, tune the threshold on `source_val` and compute the metrics with `calculate_all_metrics`.
3. In `hyper_tuning.py`, add the import, the `Args` class, the `suggest_*` branch with names that match the attributes, and the default parameters.
4. Run the smoke test on the demo drug and then the checklist above.